# Statistical Significance Test (Wilcoxon Signed-Rank)
This notebook performs pairwise significance tests between our model and each baseline,
with Bonferroni correction for multiple comparisons.

- **Null hypothesis H₀**: The two paired distributions are identical.
- **Significance level**: $\alpha = 0.05$ (family-wise).
- **Correction**: Bonferroni-adjusted $\alpha_{\text{adj}} = 0.05 / m$, where $m$ is the number of pairwise comparisons.



In [ ]:
import os
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start the kernel inside the OtakuLab checkout.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
IS_LINUX = os.name == "posix"
from scripts.evaluation_io import (
    CHARACTERS, evaluation_paths, read_neutral_samples, read_generations,
    save_generations, sample_ids, validate_metric_pairs, align_metric_pairs, read_paired_metrics, write_run_manifest,
)

EVALUATION_VARIANT = "paper"  # "expanded": review-period 230-sentence evaluation
paths = evaluation_paths(ROOT, EVALUATION_VARIANT)
neutral_samples = read_neutral_samples(paths.test_file)
paths.run_dir.mkdir(parents=True, exist_ok=True)
print("Evaluation:", EVALUATION_VARIANT, "inputs:", len(neutral_samples))
from scipy import stats
import json
import numpy as np
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────
# H₀: The two paired distributions are identical.
# α  = 0.05 (family-wise), Bonferroni-corrected per comparison.

EVALUATE_RESULTS_PATH = paths.metric_input
TARGET_MODEL_NAME = "Ours(DPO)"
ALPHA_FAMILY = 0.05

# Baselines to compare against
BASELINE_NAMES = [
    "BaselineA",
    "BaselineB",
    "BaselineC",
]

# ── Load data ──────────────────────────────────────────────────────
# The file is a single JSON object keyed by model name.
evaluate_result = read_paired_metrics(paths)
print("Metric source:", EVALUATE_RESULTS_PATH)
print(f"Models in file: {list(evaluate_result.keys())}\n")

# Check that all required models exist
missing = []
if TARGET_MODEL_NAME not in evaluate_result:
    missing.append(TARGET_MODEL_NAME)
for bl in BASELINE_NAMES:
    if bl not in evaluate_result:
        missing.append(bl)
if missing:
    print(f"WARNING: missing models — {missing}")
    BASELINE_NAMES = [bl for bl in BASELINE_NAMES if bl not in missing]

# ── Helper ─────────────────────────────────────────────────────────

def wilcoxon_paired(name, list1, list2, alpha_adj):
    """Run Wilcoxon signed-rank test (two-sided) and print results."""
    list1 = np.asarray(list1, dtype=float)
    list2 = np.asarray(list2, dtype=float)

    stat, p_value = stats.wilcoxon(list1, list2, alternative='two-sided')

    # Compute the approximate Z-based effect size r = Z / sqrt(n).
    d = list1 - list2
    nonzero = d != 0
    d = d[nonzero]
    n = len(d)

    if n == 0:
        Z = 0.0
        r = 0.0
    else:
        ranks = stats.rankdata(np.abs(d))
        W_plus = np.sum(ranks[d > 0])
        mu_W = n * (n + 1) / 4
        sigma_W = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)
        Z = (W_plus - mu_W + 0.5 * np.sign(mu_W - W_plus)) / sigma_W
        r = Z / np.sqrt(n)

    significant = p_value < alpha_adj
    return {
        "stat": stat,
        "p_value": p_value,
        "Z": Z,
        "r": r,
        "n": len(list1),
        "n_nonzero": n,
        "significant": significant,
        "alpha_adj": alpha_adj,
    }

# ── Pairwise comparisons ───────────────────────────────────────────
# Two metrics: Semantic Score, Valid Style Score
# Bonferroni: m = len(BASELINE_NAMES) × 2 (one per metric)
N_COMPARISONS = len(BASELINE_NAMES) * 2
alpha_adj = ALPHA_FAMILY / N_COMPARISONS

print("=" * 72)
print("Wilcoxon Signed-Rank — Pairwise Tests")
print(f"H₀  : distributions are identical")
print(f"α   = {ALPHA_FAMILY} (family-wise)")
print(f"α′  = {alpha_adj:.6f} (Bonferroni-adjusted, m = {N_COMPARISONS})")
print(f"H₁  : distributions differ (two-sided)")
print("=" * 72)

summary = []

for bl_name in BASELINE_NAMES:
    for metric, metric_key in [("Semantic Score", "semantic"),
                                ("Valid Style Score", "valid_style")]:
        target_scores = evaluate_result[TARGET_MODEL_NAME][metric_key]
        baseline_scores = evaluate_result[bl_name][metric_key]
        res = wilcoxon_paired(
            f"{TARGET_MODEL_NAME} vs {bl_name} — {metric}",
            target_scores, baseline_scores,
            alpha_adj,
        )
        res["baseline"] = bl_name
        res["metric"] = metric
        summary.append(res)

        sig_str = "✓ SIGNIFICANT" if res["significant"] else "✗ NS"
        print(f"\n{res['baseline']} — {res['metric']}")
        print(f"  N = {res['n']} pairs ({res['n_nonzero']} non-zero)")
        print(f"  Z = {res['Z']:.4f}, p = {res['p_value']:.6f}")
        print(f"  r = {res['r']:.4f}  (Z / sqrt(n))")
        print(f"  p < α′ ({res['alpha_adj']:.6f})?  →  {sig_str}")

print("\n" + "=" * 72)
print("SUMMARY TABLE")
print("=" * 72)
print(f"{'Baseline':<14} {'Metric':<20} {'Z':>8} {'p':>10} {'r':>8} {'Sig?':>6}")
print("-" * 72)
for s in summary:
    bl = s["baseline"]
    m = s["metric"]
    sig = "✓" if s["significant"] else ""
    print(f"{bl:<14} {m:<20} {s['Z']:>8.4f} {s['p_value']:>10.6f} {s['r']:>8.4f} {sig:>6}")
